# NHANES 2005–2008 replication: retinopathy and time-dependent mortality (pre-registered)

**Purpose.** Independent replication, in a second nationally representative US cohort examined 17 years later,
of the NHANES III (1988–1994) discovery finding: retinopathy on fundus photographs was associated with higher
mortality during the first decade after the examination, and the association was absent thereafter.

**Pre-registration.** The analysis plan is registered on OSF **before** any NHANES 2005–2008 data are
downloaded. This notebook refuses to run until the registration link is saved in a text file named
`osf_registration.txt` in this folder (the notebook file itself is never edited, so its code hash stays equal to
the one stated in the registration; the hash is recomputed and logged at run time).

**Data (all public, no application needed).** NHANES 2005–2006 (suffix `_D`) and 2007–2008 (`_E`) SAS transport
files: DEMO, OPXRET (retinal imaging, two 45° non-mydriatic fields per eye, ETDRS-based grading), BMX, BPX, BPQ,
DIQ, GHB, TCHOL, HDL, BIOPRO, SMQ, MCQ; NCHS 2019 public-use linked mortality files (follow-up to 31 Dec 2019).

**Analyses** (roles as registered):

| Role | Analysis |
|---|---|
| Primary (H1) | Model 2 HR for retinopathy (harmonized definition) and all-cause death during 0–10 y |
| Secondary (H1b) | Same for CVD death |
| Secondary (H2) | Window HRs 0–5, 5–10, >10 y; test of HR(0–10 y) vs HR(>10 y) and 3-window heterogeneity |
| Secondary | Standardized 5- and 10-y risk differences, 10-y RMST / RMTL differences, 10-y ΔC; whole-follow-up HRs |
| Secondary | Cross-cohort fixed-effect pooling with NHANES III (first-decade HRs) and between-cohort Q test |
| Sensitivity | NCHS definition (levels ≥14), levels ≥20, single-eye assessment, 2–10 y, excluding prior MI, unweighted |
| Exploratory | Severity, diabetes strata and interaction |

**Run order.** (1) register the plan on OSF, (2) save the link in `osf_registration.txt`, (3) Run All.
Outputs go to `output_0508/`; raw files to `data_0508/`. The NHANES III outputs in `output/` are read (never
changed) for the cross-cohort comparison.

In [4]:
# ------------------------------------------------------------------ 0. Setup and pre-registration guard
import os, sys, json, hashlib, warnings, platform, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
import nhanes3_tools as nt
import lifelines
for _f in ("read_xpt", "fetch_nhanes_cont", "survey_jackknife", "absolute_risk_suite", "pool_fixed_effect"):
    if not hasattr(nt, _f):
        raise ImportError("nhanes3_tools.py is older than this notebook: replace it with the v9 file delivered with it")

DATA_DIR = HERE / "data_0508"
OUT_DIR = HERE / "output_0508"
FIG_DIR = OUT_DIR / "figures"
NH3_OUT = HERE / "output"                      # NHANES III discovery outputs (read only)
NB_FILE = HERE / "nhanes0508_replication.ipynb"
for _d in (DATA_DIR, OUT_DIR, FIG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

PREREG_FILE = HERE / "osf_registration.txt"
TEST_MODE = os.environ.get("NHANES_REPLICATION_TEST_MODE") == "1"     # synthetic-file code tests only
prereg_url = PREREG_FILE.read_text(encoding="utf-8").strip() if PREREG_FILE.exists() else ""
if not prereg_url.lower().startswith(("https://osf.io/", "http://osf.io/", "https://doi.org/10.17605/osf.io/")):
    if not TEST_MODE:
        raise RuntimeError("Pre-registration link not found. Register the analysis plan on OSF first, then save the "
                           "registration URL (https://osf.io/...) as the only line of osf_registration.txt in this folder.")
    print("TEST MODE: running without a registration link (synthetic files only)")

def _code_sha256(nb_path):
    # hash of the code cells only: unaffected by outputs, execution counts or editor metadata
    nbj = json.loads(Path(nb_path).read_text(encoding="utf-8"))
    src = "\n".join(("".join(c["source"]) if isinstance(c["source"], list) else c["source"])
                    for c in nbj["cells"] if c["cell_type"] == "code")
    return hashlib.sha256(src.replace("\r\n", "\n").encode("utf-8")).hexdigest()

warnings.filterwarnings("ignore", category=FutureWarning)
from lifelines.exceptions import StatisticalWarning
warnings.filterwarnings("ignore", category=StatisticalWarning)
pd.set_option("display.width", 160, "display.max_columns", 40, "display.max_rows", 200)
nt.set_figure_style()
RUNLOG = {"notebook": NB_FILE.name, "osf_registration": prereg_url or None, "test_mode": TEST_MODE,
          "notebook_code_sha256": _code_sha256(NB_FILE) if NB_FILE.exists() else None,
          "tools_sha256": hashlib.sha256((HERE / "nhanes3_tools.py").read_bytes().replace(b"\r\n", b"\n")).hexdigest(),
          "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
          "lifelines": lifelines.__version__}
print(json.dumps(RUNLOG, indent=2))

{
  "notebook": "nhanes0508_replication.ipynb",
  "osf_registration": "https://osf.io/j78ym",
  "test_mode": false,
  "notebook_code_sha256": "03febf8068a36bf7abd9f97e97d9ce94e99e8c69204c3c3e7e4b4e46b0674d83",
  "tools_sha256": "9fcdbfbd2c75436216dc355927a1e12c964df24daa141193b273f621c489e9a3",
  "python": "3.12.10",
  "pandas": "2.3.3",
  "numpy": "2.4.4",
  "lifelines": "0.30.3"
}


## CONFIG — files, variables and coding decisions (as registered)

In [5]:
CYCLES = ["D", "E"]                      # D = 2005-2006, E = 2007-2008
COMPONENTS = {
    "DEMO":   ["SEQN", "RIDSTATR", "RIDAGEYR", "RIAGENDR", "RIDRETH1", "DMDEDUC2", "WTMEC2YR", "SDMVPSU", "SDMVSTRA"],
    "OPXRET": ["SEQN", "OPDURET", "OPDURL4", "OPDUARMA", "OPDDRET", "OPDSRET"],
    "BMX":    ["SEQN", "BMXBMI"],
    "BPX":    ["SEQN", "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4", "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"],
    "BPQ":    ["SEQN", "BPQ020", "BPQ040A", "BPQ050A"],
    "DIQ":    ["SEQN", "DIQ010", "DIQ050", "DID070"],
    "GHB":    ["SEQN", "LBXGH"],
    "TCHOL":  ["SEQN", "LBXTC"],
    "HDL":    ["SEQN", "LBDHDD"],
    "BIOPRO": ["SEQN", "LBXSCR"],
    "SMQ":    ["SEQN", "SMQ020", "SMQ040"],
    "MCQ":    ["SEQN", "MCQ160B", "MCQ160C", "MCQ160E", "MCQ160F"],
}

# Codebook frequencies (CDC OPXRET_D / OPXRET_E documentation) - exact integrity checks on the downloaded files.
# For 2007-2008 only the transcribed OPDURET levels are checked.
CODEBOOK_COUNTS = {
    "D": {"OPDUARMA": {0: 2145, 1: 314}, "OPDURL4": {1: 2145, 2: 244, 3: 57, 4: 13},
          "OPDURET": {10: 2022, 12: 33, 13: 90, 14: 11, 15: 57, 20: 129, 31: 47, 41: 52, 51: 5, 60: 3,
                      61: 0, 62: 0, 63: 1, 64: 1, 65: 7, 70: 1, 80: 0}},
    "E": {"OPDUARMA": {0: 2849, 1: 396}, "OPDURL4": {1: 2849, 2: 329, 3: 49, 4: 18},
          "OPDURET": {10: 2589, 12: 80, 13: 180}},
}
# External benchmark: Zhang et al., JAMA 2010;304:649-656 (NHANES 2005-2008, adults >= 40 with diabetes defined as
# self-report or HbA1c >= 6.5%): diabetic retinopathy 28.5% (95% CI 24.9-32.5), weighted.
BENCHMARK_DR_IN_DIABETES = (28.5, 24.9, 32.5)

# Retinopathy, worse eye (OPDURET; ETDRS-based levels): 10 none; 12 non-diabetic retinal disease specific
# retinopathy; 13 questionable; 14 HE/SE/IRMA/VL without MA; 15 hemorrhage only; 20 MA only; 31 early NPR;
# 41 moderate NPR; 51 severe NPR; 60 fibrous proliferation only; 61-64 PRP laser scars (with none/MA/early/
# moderate-severe NPR); 65 PR < HRC; 70 PR >= HRC; 80 total vitreous hemorrhage.
RET_LEVELS_VALID = [10, 12, 13, 14, 15, 20, 31, 41, 51, 60, 61, 62, 63, 64, 65, 70, 80]
#   PRIMARY  "ret_any"     level 12 or >= 14  (harmonized with the NHANES III discovery definition 'any_ge12')
#   SECONDARY "ret_any_ge14" levels >= 14 (NCHS OPDUARMA)       "ret_any_ge20" levels >= 20 (MA or worse)
#   SEVERITY  none (10, 13); mild (12, 14-31); moderate/severe NPR (41-51); proliferative or laser (60-80)
#   SINGLE EYE  eye chosen by SEQN parity (even -> right OPDDRET, odd -> left OPDSRET), same rule as primary;
#               persons whose selected eye has no gradable level are excluded from that analysis only.

# Covariates harmonized with NHANES III Model 2 (education is categorical here: DMDEDUC2):
#   race: RIDRETH1 3 -> 1 non-Hispanic White (reference), 4 -> 2 non-Hispanic Black, 1 -> 3 Mexican American,
#         2 or 5 -> 4 other (incl. other Hispanic, multiracial)  [NHANES III DMARETHN coding]
#   education: DMDEDUC2 1-2 < high school (reference), 3 high school/GED, 4-5 > high school; 7/9 -> missing
#   SBP / DBP: mean of available readings 1-4 (DBP 0 -> missing)
#   BP medication: BPQ050A = 1; diabetes: DIQ010 = 1 or HbA1c >= 6.5% or DIQ050 = 1 or DID070 = 1
#   smoking: SMQ020 = 2 never; SMQ020 = 1 & SMQ040 in {1, 2} current; SMQ020 = 1 & SMQ040 = 3 former
#   prior MI: MCQ160E = 1
#   creatinine: 2005-2006 recalibrated per the CDC BIOPRO_D note: -0.016 + 0.978 x LBXSCR; 2007-2008 as released
#   eGFR: CKD-EPI 2021 (race-free), as in NHANES III
YES, NO = 1, 2
CREATININE_CAL = {"D": (-0.016, 0.978), "E": None}
CVD_CODES = {1, 5}                      # UCOD_LEADING: diseases of heart; cerebrovascular diseases

# Pre-registered analysis constants
AGE_MIN = 40
FIRST_DECADE = (0, 10)
WINDOWS = [(0, 5), (5, 10), (10, 40)]   # the last window runs to the end of follow-up (max ~14.9 y)
HORIZONS = [5, 10]                      # standardized risks (years); 10 y < shortest potential follow-up (~11 y)
TAUS = [10]
C_HORIZON = 10
CURVE_STEP, CURVE_MAX = 1 / 12, 10     # monthly points for the standardized curves
MIN_FOLLOWUP_YEARS_SENS = 2.0

## 1. Download the public-use files (first run only)

In [6]:
paths = nt.fetch_nhanes_cont(DATA_DIR, COMPONENTS.keys(), CYCLES)
print("\nCodebooks:")
for sfx in CYCLES:
    for comp in COMPONENTS:
        print(f"  {comp}_{sfx}: {nt.nhanes_cont_url(comp, sfx, 'htm')}")

[get ] https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/DEMO_D.xpt
      -> DEMO_D.xpt: 3.6 MB in 5 s
[get ] https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/OPXRET_D.xpt
      -> OPXRET_D.xpt: 5.2 MB in 4 s
[get ] https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/BMX_D.xpt
      -> BMX_D.xpt: 2.2 MB in 4 s
[get ] https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/BPX_D.xpt
      -> BPX_D.xpt: 2.2 MB in 4 s
[get ] https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/BPQ_D.xpt
      -> BPQ_D.xpt: 0.8 MB in 2 s
[get ] https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/DIQ_D.xpt
      -> DIQ_D.xpt: 2.8 MB in 4 s
[get ] https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/GHB_D.xpt
      -> GHB_D.xpt: 0.1 MB in 2 s
[get ] https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/TCHOL_D.xpt
      -> TCHOL_D.xpt: 0.2 MB in 2 s
[get ] https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/HDL_D.xpt
      -> HDL_D.xpt: 0.2

## 2. Load, integrity checks against the codebooks, merge, sample flow

In [7]:
frames, integrity = [], []
for sfx in CYCLES:
    parts = {comp: nt.read_xpt(paths[sfx][comp], cols) for comp, cols in COMPONENTS.items()}
    for comp, fr in parts.items():
        assert fr["SEQN"].is_unique, f"SEQN not unique in {comp}_{sfx}"
    # exact codebook frequencies of the retinal-imaging variables
    for var, expected in CODEBOOK_COUNTS[sfx].items():
        got = parts["OPXRET"][var].value_counts()
        for code_, n_exp in expected.items():
            n_got = int(got.get(code_, 0))
            integrity.append({"cycle": sfx, "variable": var, "code": code_, "codebook": n_exp, "file": n_got,
                              "match": n_got == n_exp})
    m = parts["DEMO"]
    for comp in COMPONENTS:
        if comp != "DEMO":
            m = m.merge(parts[comp], on="SEQN", how="left")
    mort = nt.read_lmf(paths[sfx]["MORT"], expected_seqn=parts["DEMO"]["SEQN"])
    m = m.merge(mort, on="SEQN", how="left", indicator="mort_merge")
    m["has_mort"] = (m["mort_merge"] == "both").astype(int)
    m["cycle"] = sfx
    frames.append(m.drop(columns="mort_merge"))
    print(f"{sfx}: DEMO {len(parts['DEMO']):,} | OPXRET {len(parts['OPXRET']):,} | LMF {len(mort):,}")
integrity = pd.DataFrame(integrity)
integrity.to_csv(OUT_DIR / "codebook_integrity_checks.csv", index=False)
print(integrity.groupby(["cycle", "variable"])["match"].all())
if not integrity["match"].all():
    print(integrity[~integrity["match"]].to_string(index=False))
    raise RuntimeError("Downloaded OPXRET frequencies differ from the CDC codebook - stop and report (variable mapping)")
df = pd.concat(frames, ignore_index=True)
assert df["SEQN"].is_unique

flow = []
def step(label, mask):
    flow.append((label, int(mask.sum())))
    print(f"{label:62s} n = {int(mask.sum()):6d}")
    return mask
m = step("NHANES 2005-2008 participants (DEMO)", pd.Series(True, index=df.index))
m = step(f"Age >= {AGE_MIN} years at screening", m & (df["RIDAGEYR"] >= AGE_MIN))
m = step("MEC-examined (RIDSTATR = 2, WTMEC2YR > 0)", m & (df["RIDSTATR"] == 2) & (df["WTMEC2YR"] > 0))
m = step("Retinopathy level graded in >= 1 eye (OPDURET)", m & df["OPDURET"].isin(RET_LEVELS_VALID))
m = step("Mortality-linkage eligible (ELIGSTAT = 1)", m & (df["ELIGSTAT"] == 1))
m = step("Follow-up time present (PERMTH_EXM)", m & df["PERMTH_EXM"].notna())
cohort_mask = m.copy()
pd.DataFrame(flow, columns=["step", "n"]).to_csv(OUT_DIR / "sample_flow.csv", index=False)

[LMF] NHANES_2005_2006_MORT_2019_PUBLIC.dat: 10,348 records; SEQN range 31127-41474
[LMF] columns 7-14 non-blank in 0 record(s) (expected 0 for NHANES)
[LMF] ELIGSTAT counts: {1: 5561, 2: 4785, 3: 2}
[LMF] MORTSTAT among ELIGSTAT = 1: {0.0: 4534, 1.0: 1027}
[LMF] UCOD_LEADING among deaths: {1.0: 264, 2.0: 213, 3.0: 61, 4.0: 30, 5.0: 64, 6.0: 39, 7.0: 32, 8.0: 20, 9.0: 28, 10.0: 276}
[LMF] PERMTH_EXM (months) among ELIGSTAT = 1: min 1, median 165, max 180
[LMF] linkage: 100.00% of 10,348 expected participants found in LMF
D: DEMO 10,348 | OPXRET 2,934 | LMF 10,348
[LMF] NHANES_2007_2008_MORT_2019_PUBLIC.dat: 10,149 records; SEQN range 41475-51623
[LMF] columns 7-14 non-blank in 0 record(s) (expected 0 for NHANES)
[LMF] ELIGSTAT counts: {1: 6219, 2: 3921, 3: 9}
[LMF] MORTSTAT among ELIGSTAT = 1: {0.0: 5093, 1.0: 1126}
[LMF] UCOD_LEADING among deaths: {1.0: 267, 2.0: 256, 3.0: 67, 4.0: 28, 5.0: 69, 6.0: 37, 7.0: 39, 8.0: 27, 9.0: 25, 10.0: 311}
[LMF] PERMTH_EXM (months) among ELIGSTAT = 1

## 3. Derived variables (exposures, covariates, outcomes, survey design)

In [8]:
d = df[cohort_mask].copy()
lvl = d["OPDURET"]
d["ret_any"] = (lvl.isin([12]) | (lvl >= 14)).astype(int)                     # PRIMARY (harmonized)
d["ret_any_ge14"] = (lvl >= 14).astype(int)                                    # NCHS OPDUARMA
d["ret_any_ge20"] = (lvl >= 20).astype(int)
assert (d["ret_any_ge14"] == d["OPDUARMA"]).all(), "levels >= 14 must reproduce OPDUARMA"
d["ret_score"] = np.select([lvl >= 60, lvl.between(41, 51), lvl.isin([12]) | lvl.between(14, 31)], [3, 2, 1], default=0)
d["ret_mild"] = (d["ret_score"] == 1).astype(float)
d["ret_modplus"] = (d["ret_score"] >= 2).astype(float)
eye_lvl = np.where(d["SEQN"] % 2 == 0, d["OPDDRET"], d["OPDSRET"])
eye_lvl = pd.Series(eye_lvl, index=d.index).where(lambda s: s.isin(RET_LEVELS_VALID))
d["ret_any_1eye"] = np.where(eye_lvl.isna(), np.nan, ((eye_lvl == 12) | (eye_lvl >= 14)).astype(float))

d["age"] = d["RIDAGEYR"]
d["female"] = (d["RIAGENDR"] == 2).astype(int)
d["race_eth"] = d["RIDRETH1"].map({3: 1, 4: 2, 1: 3, 2: 4, 5: 4})
for lev in (2, 3, 4):
    d[f"race_eth_{lev}"] = np.where(d["race_eth"].isna(), np.nan, (d["race_eth"] == lev).astype(float))
edu = d["DMDEDUC2"].map({1: 0, 2: 0, 3: 1, 4: 2, 5: 2})
d["edu_hs"] = np.where(edu.isna(), np.nan, (edu == 1).astype(float))
d["edu_gt_hs"] = np.where(edu.isna(), np.nan, (edu == 2).astype(float))
d["bmi"] = d["BMXBMI"].where(d["BMXBMI"].between(10, 80))
_sy = d[["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]]
_di = d[["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]]
d["sbp"] = _sy.where((_sy >= 60) & (_sy <= 300)).mean(axis=1)
d["dbp"] = _di.where((_di >= 20) & (_di <= 200)).mean(axis=1)        # diastolic 0 readings -> missing
d["hba1c"] = d["LBXGH"].where(d["LBXGH"].between(2, 20))
d["tc"] = d["LBXTC"].where(d["LBXTC"].between(50, 800))
d["hdl"] = d["LBDHDD"].where(d["LBDHDD"].between(5, 200))
scr = d["LBXSCR"].where(d["LBXSCR"].between(0.1, 20))
for sfx, cal in CREATININE_CAL.items():
    if cal:
        sel = d["cycle"] == sfx
        scr[sel] = cal[0] + cal[1] * scr[sel]
d["egfr"] = nt.egfr_ckdepi2021(scr, d["age"], d["female"] == 1)
d["bp_meds"] = (d["BPQ050A"] == YES).astype(int)
d["dm_told"] = (d["DIQ010"] == YES).astype(int)
d["diabetes"] = ((d["dm_told"] == 1) | (d["hba1c"] >= 6.5) | (d["DIQ050"] == YES) | (d["DID070"] == YES)).astype(int)
d["diabetes_zhang"] = ((d["dm_told"] == 1) | (d["hba1c"] >= 6.5)).astype(int)        # benchmark definition only
d["hypertension"] = ((d["sbp"] >= 140) | (d["dbp"] >= 90) | (d["bp_meds"] == 1)).astype(int)
d["smoke"] = pd.Series(np.select([d["SMQ020"] == NO, (d["SMQ020"] == YES) & d["SMQ040"].isin([1, 2]),
                                  (d["SMQ020"] == YES) & (d["SMQ040"] == 3)], ["never", "current", "former"],
                                 default="missing"), index=d.index).replace({"missing": np.nan})
for lev in ("former", "current"):
    d[f"smoke_{lev}"] = np.where(d["smoke"].isna(), np.nan, (d["smoke"] == lev).astype(float))
d["prior_mi"] = (d["MCQ160E"] == YES).astype(int)

d["time_y"] = d["PERMTH_EXM"] / 12.0
d["death_all"] = (d["MORTSTAT"] == 1).astype(int)
d["death_cvd"] = ((d["MORTSTAT"] == 1) & d["UCOD_LEADING"].isin(CVD_CODES)).astype(int)
d.loc[d["time_y"] <= 0, "time_y"] = 0.5 / 12.0          # same half-month convention as NHANES III

d["w"] = d["WTMEC2YR"] / len(CYCLES)                    # combined 4-year MEC weight (NCHS analytic guidelines)
d["psu"] = d["SDMVSTRA"].astype(int).astype(str) + "_" + d["SDMVPSU"].astype(int).astype(str)
alld = df[(df["RIDSTATR"] == 2) & (df["WTMEC2YR"] > 0)].copy()
alld["psu"] = alld["SDMVSTRA"].astype(int).astype(str) + "_" + alld["SDMVPSU"].astype(int).astype(str)
DESIGN_PSUS = alld[["SDMVSTRA", "psu"]].drop_duplicates()        # every PSU of the examined sample (domain estimation)
SVY = dict(cluster="psu", strata_col="SDMVSTRA", design_psus=DESIGN_PSUS)
print(f"Design: {DESIGN_PSUS['SDMVSTRA'].nunique()} strata, {len(DESIGN_PSUS)} PSUs")

fu_rkm = lifelines.KaplanMeierFitter().fit(d["time_y"], 1 - d["death_all"]).median_survival_time_
print(f"Analytic cohort n = {len(d):,}; deaths = {d['death_all'].sum():,}; CVD deaths = {d['death_cvd'].sum():,}; "
      f"median follow-up (reverse KM) = {fu_rkm:.1f} y; max = {d['time_y'].max():.1f} y")

defs = [("Level 12 or >= 14 (harmonized with NHANES III; primary)", "ret_any"),
        ("Levels >= 14 (NCHS OPDUARMA)", "ret_any_ge14"), ("Levels >= 20", "ret_any_ge20"),
        ("Single eye, level 12 or >= 14", "ret_any_1eye")]
rows = []
for lab, col in defs:
    s = d[d[col].notna()]
    rows.append({"definition": lab, "column": col, "n_assessed": len(s), "n_exposed": int(s[col].sum()),
                 "weighted_pct_all": round(100 * nt.wprop(s[col], s["w"]), 2),
                 "weighted_pct_no_diabetes": round(100 * nt.wprop(s.loc[s["diabetes"] == 0, col], s.loc[s["diabetes"] == 0, "w"]), 2),
                 "weighted_pct_diabetes": round(100 * nt.wprop(s.loc[s["diabetes"] == 1, col], s.loc[s["diabetes"] == 1, "w"]), 2)})
ret_defs = pd.DataFrame(rows)
ret_defs.to_csv(OUT_DIR / "retinopathy_definitions.csv", index=False)
print(ret_defs.to_string(index=False))
bz = 100 * nt.wprop(d.loc[d["diabetes_zhang"] == 1, "ret_any_ge14"], d.loc[d["diabetes_zhang"] == 1, "w"])
print(f"Benchmark: levels >= 14 among diabetes (self-report or HbA1c >= 6.5%) = {bz:.1f}% "
      f"(Zhang 2010: {BENCHMARK_DR_IN_DIABETES[0]}%, 95% CI {BENCHMARK_DR_IN_DIABETES[1]}-{BENCHMARK_DR_IN_DIABETES[2]})")
RUNLOG.update({"n_cohort": len(d), "deaths_all": int(d["death_all"].sum()), "deaths_cvd": int(d["death_cvd"].sum()),
               "median_followup_reverse_km_years": float(fu_rkm), "benchmark_dr_in_diabetes_pct": round(bz, 1)})
d.to_csv(OUT_DIR / "analytic_dataset_0508.csv", index=False)

Design: 31 strata, 62 PSUs
Analytic cohort n = 5,703; deaths = 1,449; CVD deaths = 440; median follow-up (reverse KM) = 12.8 y; max = 15.0 y
                                             definition       column  n_assessed  n_exposed  weighted_pct_all  weighted_pct_no_diabetes  weighted_pct_diabetes
Level 12 or >= 14 (harmonized with NHANES III; primary)      ret_any        5703        823             11.19                      7.97                  30.82
                           Levels >= 14 (NCHS OPDUARMA) ret_any_ge14        5703        710              9.75                      6.70                  28.31
                                           Levels >= 20 ret_any_ge20        5703        553              7.46                      4.48                  25.64
                          Single eye, level 12 or >= 14 ret_any_1eye        5567        554              7.40                      4.57                  24.88
Benchmark: levels >= 14 among diabetes (self-report or HbA1c >= 

## 4. Table 1 (survey-weighted) by retinopathy status

In [9]:
t1 = nt.table1(d, by="ret_any", w="w",
               continuous=["age", "bmi", "sbp", "dbp", "hba1c", "tc", "hdl", "egfr"],
               binary=["female", "diabetes", "hypertension", "bp_meds", "smoke_current", "prior_mi", "edu_gt_hs"],
               categorical=["race_eth"])
t1.to_csv(OUT_DIR / "table1_by_retinopathy.csv", index=False)
t1

,variable,type,0,1
0,n (unweighted),,4880,823
1,age,mean (SD),55.9 (11.6),61.0 (12.3)
2,bmi,mean (SD),29.0 (6.4),29.8 (6.3)
3,sbp,mean (SD),125.5 (17.9),132.5 (22.4)
4,dbp,mean (SD),72.7 (11.3),70.8 (13.4)
5,hba1c,mean (SD),5.6 (0.7),6.4 (1.6)
6,tc,mean (SD),204.9 (41.1),193.8 (44.8)
7,hdl,mean (SD),54.1 (16.6),52.3 (15.0)
8,egfr,mean (SD),88.5 (17.7),82.5 (22.3)
9,female,weighted %,53.6,44.6


## 5. Pre-registered Cox analyses (design-based variance: strata + PSU, Taylor linearization)

In [10]:
X = d.copy()
M1 = ["age", "female", "race_eth_2", "race_eth_3", "race_eth_4"]
M2 = M1 + ["edu_hs", "edu_gt_hs", "bmi", "sbp", "bp_meds", "diabetes", "hba1c", "tc", "hdl", "egfr", "prior_mi",
           "smoke_former", "smoke_current"]
M2 = [c for c in M2 if X[c].nunique(dropna=True) > 1]                 # a category absent from the data is dropped
RUNLOG["covariates_model2"] = M2

def period_frame(data, outcome, a, b):
    # persons alive at the start of the period, time re-origined, censored at the period end (= left truncation)
    P = data[data["time_y"] > a].copy()
    P["t_p"] = P["time_y"].clip(upper=b) - a
    P["ev_p"] = ((P[outcome] == 1) & (P["time_y"] <= b)).astype(int)
    return P

from lifelines.statistics import proportional_hazard_test
res_rows, errors = [], []
def fit(role, analysis, outcome, data, terms, covs, tcol="t_p", ecol="ev_p"):
    try:
        _, res = nt.cox_weighted(data, tcol, ecol, terms + covs, "w", **SVY)
    except Exception as e:
        errors.append({"analysis": analysis, "outcome": outcome, "error": str(e)[:200]})
        return None
    for t in terms:
        r = res.loc[t]
        res_rows.append({"role": role, "analysis": analysis, "outcome": outcome, "term": t, "HR": r["HR"],
                         "HR_lo95": r["HR_lo95"], "HR_hi95": r["HR_hi95"], "p": r["p"], "n": res.attrs["n"],
                         "events": res.attrs["events"]})
    return res

tests = []
for outcome in ["death_all", "death_cvd"]:
    role = "primary (H1)" if outcome == "death_all" else "secondary (H1b)"
    FD = period_frame(X, outcome, *FIRST_DECADE)
    r0 = fit(role, "first decade, M2", outcome, FD, ["ret_any"], M2)
    fit("secondary", "first decade, M1", outcome, FD, ["ret_any"], M1)
    # H2: windows and tests
    betas, infl, win_fit = [], [], {}
    for a, b in WINDOWS:
        r = fit("secondary (H2)", f"window {a}-{b if b < 40 else 'end'} y, M2", outcome, period_frame(X, outcome, a, b), ["ret_any"], M2)
        win_fit[(a, b)] = r
        if r is not None:
            betas.append(r.loc["ret_any", "beta"]); infl.append(r.attrs["influence"]["ret_any"])
    if len(betas) == len(WINDOWS):
        het = nt.design_wald(betas, infl, X["SDMVSTRA"], X["psu"], np.array([[1, -1, 0], [0, 1, -1]]), design_psus=DESIGN_PSUS)
        tests.append({"role": "secondary (H2)", "test": "heterogeneity across 3 windows (2 df)", "outcome": outcome,
                      "chi2": het["chi2"], "df": het["df"], "p": het["p"]})
    r1 = win_fit.get((10, 40))                                   # the >10 y window model
    if r0 is not None and r1 is not None:
        con = nt.design_wald([r0.loc["ret_any", "beta"], r1.loc["ret_any", "beta"]],
                             [r0.attrs["influence"]["ret_any"], r1.attrs["influence"]["ret_any"]],
                             X["SDMVSTRA"], X["psu"], np.array([[1, -1]]), design_psus=DESIGN_PSUS)
        tests.append({"role": "secondary (H2)", "test": "log HR 0-10 y minus log HR >10 y (1 df)", "outcome": outcome,
                      "chi2": con["chi2"], "df": con["df"], "p": con["p"],
                      "ratio_of_HRs": float(np.exp(r0.loc["ret_any", "beta"] - r1.loc["ret_any", "beta"]))})
    # whole follow-up
    fit("secondary", "whole follow-up, M1", outcome, X, ["ret_any"], M1, tcol="time_y", ecol=outcome)
    fit("secondary", "whole follow-up, M2", outcome, X, ["ret_any"], M2, tcol="time_y", ecol=outcome)
    dd = X[["time_y", outcome, "ret_any"] + M2].dropna()
    cph = lifelines.CoxPHFitter().fit(dd, duration_col="time_y", event_col=outcome)
    ph = proportional_hazard_test(cph, dd, time_transform="rank")
    tests.append({"role": "secondary", "test": "Schoenfeld, rank time (unweighted), whole follow-up M2", "outcome": outcome,
                  "chi2": float(ph.summary.loc["ret_any", "test_statistic"]), "df": 1, "p": float(ph.summary.loc["ret_any", "p"])})

primary = pd.DataFrame(res_rows)
tests = pd.DataFrame(tests)
primary.to_csv(OUT_DIR / "replication_cox.csv", index=False)
tests.to_csv(OUT_DIR / "replication_tests.csv", index=False)
if errors:
    pd.DataFrame(errors).to_csv(OUT_DIR / "fit_errors.csv", index=False)
    print("Models that could not be fitted:", errors)
display(primary.round(3)); display(tests.round(4))

,role,analysis,outcome,term,HR,HR_lo95,HR_hi95,p,n,events
0,primary (H1),"first decade, M2",death_all,ret_any,1.315,1.072,1.613,0.009,5346,965
1,secondary,"first decade, M1",death_all,ret_any,1.661,1.393,1.982,0.000,5703,1072
2,secondary (H2),"window 0-5 y, M2",death_all,ret_any,1.405,1.084,1.822,0.010,5346,401
3,secondary (H2),"window 5-10 y, M2",death_all,ret_any,1.254,0.956,1.645,0.103,4945,564
4,secondary (H2),"window 10-end y, M2",death_all,ret_any,1.291,0.889,1.875,0.180,4381,359
5,secondary,"whole follow-up, M1",death_all,ret_any,1.654,1.420,1.927,0.000,5703,1449
6,secondary,"whole follow-up, M2",death_all,ret_any,1.303,1.082,1.568,0.005,5346,1324
7,secondary (H1b),"first decade, M2",death_cvd,ret_any,1.376,1.013,1.867,0.041,5346,294
8,secondary,"first decade, M1",death_cvd,ret_any,1.891,1.486,2.407,0.000,5703,331
9,secondary (H2),"window 0-5 y, M2",death_cvd,ret_any,1.155,0.707,1.886,0.565,5346,138


,role,test,outcome,chi2,df,p,ratio_of_HRs
0,secondary (H2),heterogeneity across 3 windows (2 df),death_all,0.5204,2,0.7709,NaN
1,secondary (H2),log HR 0-10 y minus log HR >10 y (1 df),death_all,0.0089,1,0.9250,1.0185
2,secondary,"Schoenfeld, rank time (unweighted), whole foll...",death_all,0.3487,1,0.5549,NaN
3,secondary (H2),heterogeneity across 3 windows (2 df),death_cvd,1.5396,2,0.4631,NaN
4,secondary (H2),log HR 0-10 y minus log HR >10 y (1 df),death_cvd,0.0963,1,0.7563,0.8987
5,secondary,"Schoenfeld, rank time (unweighted), whole foll...",death_cvd,0.6698,1,0.4131,NaN


In [11]:
# Registered decision rules --------------------------------------------------------------------------------
def verdict_h1(outcome):
    r = primary[(primary["analysis"] == "first decade, M2") & (primary["outcome"] == outcome)]
    if r.empty or not np.isfinite(r["HR"].iloc[0]):
        return "not estimable"
    r = r.iloc[0]
    if r["HR_lo95"] > 1:
        return f"supported: HR {r['HR']:.2f} ({r['HR_lo95']:.2f}-{r['HR_hi95']:.2f})"
    if r["HR"] > 1:
        return f"not supported (direction consistent, CI includes 1): HR {r['HR']:.2f} ({r['HR_lo95']:.2f}-{r['HR_hi95']:.2f})"
    return f"not supported: HR {r['HR']:.2f} ({r['HR_lo95']:.2f}-{r['HR_hi95']:.2f})"

def verdict_h2(outcome):
    t = tests[(tests["test"].str.startswith("log HR 0-10")) & (tests["outcome"] == outcome)]
    if t.empty:
        return "not estimable"
    t = t.iloc[0]
    if t["ratio_of_HRs"] > 1 and t["p"] < 0.05:
        return f"supported: HR(0-10)/HR(>10) = {t['ratio_of_HRs']:.2f}, P = {t['p']:.3f}"
    if t["ratio_of_HRs"] > 1:
        return f"direction consistent, not significant: ratio {t['ratio_of_HRs']:.2f}, P = {t['p']:.3f}"
    return f"not supported: ratio {t['ratio_of_HRs']:.2f}, P = {t['p']:.3f}"

verdicts = pd.DataFrame([
    {"hypothesis": "H1 (primary): first-decade HR > 1, all-cause", "result": verdict_h1("death_all")},
    {"hypothesis": "H1b: first-decade HR > 1, CVD", "result": verdict_h1("death_cvd")},
    {"hypothesis": "H2: HR(0-10 y) > HR(>10 y), all-cause", "result": verdict_h2("death_all")},
    {"hypothesis": "H2: HR(0-10 y) > HR(>10 y), CVD", "result": verdict_h2("death_cvd")},
])
verdicts.to_csv(OUT_DIR / "replication_verdicts.csv", index=False)
RUNLOG["verdicts"] = verdicts.set_index("hypothesis")["result"].to_dict()
print(verdicts.to_string(index=False))

                                  hypothesis                                                       result
H1 (primary): first-decade HR > 1, all-cause                               supported: HR 1.31 (1.07-1.61)
               H1b: first-decade HR > 1, CVD                               supported: HR 1.38 (1.01-1.87)
       H2: HR(0-10 y) > HR(>10 y), all-cause direction consistent, not significant: ratio 1.02, P = 0.925
             H2: HR(0-10 y) > HR(>10 y), CVD                         not supported: ratio 0.90, P = 0.756


## 6. Standardized absolute risks, RMST / RMTL and incremental discrimination (jackknife CIs)

In [12]:
cc = X.dropna(subset=["time_y", "death_all", "death_cvd", "ret_any", "w"] + M2).copy()
def suite(frame):
    return nt.absolute_risk_suite(frame, "time_y", "death_all", "death_cvd", "ret_any", M2, "w",
                                  horizons=HORIZONS, taus=TAUS, c_horizon=C_HORIZON,
                                  curve_step=CURVE_STEP, curve_max=CURVE_MAX)
t0 = time.time()
jk, R = nt.survey_jackknife(suite, cc, "w", "SDMVSTRA", "psu", design_psus=DESIGN_PSUS, progress_every=10)
RUNLOG.update({"jackknife_replicates": jk.attrs["n_replicates"], "jackknife_failed": jk.attrs["failed_replicates"],
               "jackknife_minutes": round((time.time() - t0) / 60, 1)})
jk.to_csv(OUT_DIR / "jackknife_all_estimands.csv")

def fmt(k, dec=1):
    r = jk.loc[k]
    return f"{r['estimate']:.{dec}f} ({r['lo95']:.{dec}f} to {r['hi95']:.{dec}f})"
rows = []
for h in HORIZONS:
    rows.append({"outcome": "all-cause death", "years": h, "no_retinopathy_pct": fmt(f"all_F0_{h:g}"),
                 "retinopathy_pct": fmt(f"all_F1_{h:g}"), "risk_difference_pp": fmt(f"all_RD_{h:g}")})
    rows.append({"outcome": "CVD death (competing risk)", "years": h, "no_retinopathy_pct": fmt(f"cause_CIF0_{h:g}"),
                 "retinopathy_pct": fmt(f"cause_CIF1_{h:g}"), "risk_difference_pp": fmt(f"cause_RD_{h:g}")})
abs_tab = pd.DataFrame(rows)
abs_tab.to_csv(OUT_DIR / "absolute_risk_standardized.csv", index=False)
rm_tab = pd.DataFrame([{"measure": "RMST, all-cause (months)", "horizon_years": t, "no_retinopathy": fmt(f"all_RMST0_{t:g}"),
                        "retinopathy": fmt(f"all_RMST1_{t:g}"), "difference_months": fmt(f"all_dRMST_{t:g}")} for t in TAUS]
                      + [{"measure": "RMTL to CVD death (months)", "horizon_years": t, "no_retinopathy": fmt(f"cause_RMTL0_{t:g}"),
                          "retinopathy": fmt(f"cause_RMTL1_{t:g}"), "difference_months": fmt(f"cause_dRMTL_{t:g}")} for t in TAUS])
rm_tab.to_csv(OUT_DIR / "restricted_mean_times.csv", index=False)
c_tab = pd.DataFrame([{"outcome": name, "horizon_years": C_HORIZON, "C_model2": fmt(f"C_{lab}_base", 3),
                       "C_model2_plus_retinopathy": fmt(f"C_{lab}_plus", 3), "delta_C": fmt(f"dC_{lab}", 4)}
                      for lab, name in (("all", "all-cause death"), ("cause", "CVD death"))])
c_tab.to_csv(OUT_DIR / "incremental_discrimination.csv", index=False)
print(abs_tab.to_string(index=False)); print(); print(rm_tab.to_string(index=False)); print(); print(c_tab.to_string(index=False))
pts = np.round(np.arange(0.0, CURVE_MAX + 1e-9, CURVE_STEP), 4)
curves = pd.DataFrame([{"years": p_, **{f"{k}{s}": jk.loc[f"curve_{k}_{p_:g}", c] for k in
                        ("all_F0", "all_F1", "all_RD", "cause_CIF0", "cause_CIF1", "cause_RD")
                        for s, c in (("", "estimate"), ("_lo95", "lo95"), ("_hi95", "hi95"))}} for p_ in pts])
curves.to_csv(OUT_DIR / "standardized_curves.csv", index=False)

  jackknife replicate 10/62  (26 s elapsed, ~133 s left)
  jackknife replicate 20/62  (53 s elapsed, ~111 s left)
  jackknife replicate 30/62  (81 s elapsed, ~86 s left)
  jackknife replicate 40/62  (108 s elapsed, ~59 s left)
  jackknife replicate 50/62  (135 s elapsed, ~32 s left)
  jackknife replicate 60/62  (161 s elapsed, ~5 s left)
  jackknife replicate 62/62  (167 s elapsed, ~0 s left)
[jackknife] 62 replicates, 0 failed, 0 single-PSU strata skipped
                   outcome  years  no_retinopathy_pct     retinopathy_pct risk_difference_pp
           all-cause death      5    4.8 (4.1 to 5.5)    6.1 (4.8 to 7.5)   1.3 (0.1 to 2.5)
CVD death (competing risk)      5    1.6 (1.3 to 2.0)    1.8 (1.0 to 2.5)  0.1 (-0.6 to 0.9)
           all-cause death     10 12.1 (10.8 to 13.4) 14.7 (12.1 to 17.2)   2.5 (0.4 to 4.6)
CVD death (competing risk)     10    3.6 (3.0 to 4.2)    4.5 (3.1 to 6.0)  0.9 (-0.3 to 2.2)

                   measure  horizon_years         no_retinopathy         

## 7. Sensitivity and exploratory analyses (first decade, Model 2 unless stated)

In [13]:
for outcome in ["death_all", "death_cvd"]:
    FD = period_frame(X, outcome, *FIRST_DECADE)
    fit("sensitivity", "first decade, M2, levels >= 14 (NCHS)", outcome, FD, ["ret_any_ge14"], M2)
    fit("sensitivity", "first decade, M2, levels >= 20", outcome, FD, ["ret_any_ge20"], M2)
    fit("sensitivity", "first decade, M2, single eye", outcome, FD[FD["ret_any_1eye"].notna()], ["ret_any_1eye"], M2)
    fit("sensitivity", "2-10 y after examination, M2", outcome, period_frame(X, outcome, MIN_FOLLOWUP_YEARS_SENS, 10), ["ret_any"], M2)
    fit("sensitivity", "first decade, M2, excluding prior MI", outcome, FD[FD["prior_mi"] == 0], ["ret_any"],
        [c for c in M2 if c != "prior_mi"])
    FDu = FD.assign(w=1.0)
    try:
        _, res = nt.cox_weighted(FDu, "t_p", "ev_p", ["ret_any"] + M2, "w", normalize_weights=False, **SVY)
        r = res.loc["ret_any"]
        res_rows.append({"role": "sensitivity", "analysis": "first decade, M2, unweighted", "outcome": outcome,
                         "term": "ret_any", "HR": r["HR"], "HR_lo95": r["HR_lo95"], "HR_hi95": r["HR_hi95"],
                         "p": r["p"], "n": res.attrs["n"], "events": res.attrs["events"]})
    except Exception as e:
        errors.append({"analysis": "unweighted", "outcome": outcome, "error": str(e)[:200]})
    fit("exploratory", "first decade, M2, severity", outcome, FD, ["ret_mild", "ret_modplus"], M2)
    for dm, lab in [(0, "no diabetes"), (1, "diabetes")]:
        fit("exploratory", f"first decade, M2, {lab}", outcome, FD[FD["diabetes"] == dm], ["ret_any"],
            [c for c in M2 if c != "diabetes"])
    fit("exploratory", "first decade, M2, interaction x diabetes", outcome,
        FD.assign(ret_x_dm=FD["ret_any"] * FD["diabetes"]), ["ret_x_dm"], ["ret_any"] + M2)
allres = pd.DataFrame(res_rows)
allres.to_csv(OUT_DIR / "replication_cox.csv", index=False)
if errors:
    pd.DataFrame(errors).to_csv(OUT_DIR / "fit_errors.csv", index=False)
allres[allres["role"].isin(["sensitivity", "exploratory"])].round(3)

,role,analysis,outcome,term,HR,HR_lo95,HR_hi95,p,n,events
14,sensitivity,"first decade, M2, levels >= 14 (NCHS)",death_all,ret_any_ge14,1.306,1.026,1.661,0.030,5346,965
15,sensitivity,"first decade, M2, levels >= 20",death_all,ret_any_ge20,1.296,0.987,1.704,0.062,5346,965
16,sensitivity,"first decade, M2, single eye",death_all,ret_any_1eye,1.366,1.087,1.717,0.007,5223,912
17,sensitivity,"2-10 y after examination, M2",death_all,ret_any,1.253,0.986,1.592,0.065,5216,835
18,sensitivity,"first decade, M2, excluding prior MI",death_all,ret_any,1.436,1.173,1.758,0.000,5013,805
19,sensitivity,"first decade, M2, unweighted",death_all,ret_any,1.271,1.083,1.493,0.003,5346,965
20,exploratory,"first decade, M2, severity",death_all,ret_mild,1.242,1.004,1.536,0.046,5346,965
21,exploratory,"first decade, M2, severity",death_all,ret_modplus,2.019,1.294,3.152,0.002,5346,965
22,exploratory,"first decade, M2, no diabetes",death_all,ret_any,1.347,1.084,1.674,0.007,4305,680
23,exploratory,"first decade, M2, diabetes",death_all,ret_any,1.272,0.862,1.878,0.225,1041,285


## 8. Cross-cohort comparison with NHANES III (reads `output/` from the discovery notebook)

In [14]:
cross, win_rows = [], []
tv_file = NH3_OUT / "time_varying_analyses.csv"
if tv_file.exists():
    tv3 = pd.read_csv(tv_file)
    pairs = [("first decade, M2", "ret_any", "first decade, M2", "ret_any", "primary definitions"),
             ("first decade, M2, ret_any_ge14", "ret_any_ge14", "first decade, M2, levels >= 14 (NCHS)", "ret_any_ge14",
              "levels >= 14 in both cohorts")]
    for outcome in ["death_all", "death_cvd"]:
        for a3, t3, a8, t8, lab in pairs:
            r3 = tv3[(tv3["analysis"] == a3) & (tv3["outcome"] == outcome) & (tv3["term"] == t3)]
            r8 = allres[(allres["analysis"] == a8) & (allres["outcome"] == outcome) & (allres["term"] == t8)]
            if r3.empty or r8.empty or not np.isfinite(r8["HR"].iloc[0]):
                continue
            r3, r8 = r3.iloc[0], r8.iloc[0]
            b = [np.log(r3["HR"]), np.log(r8["HR"])]
            s = [nt.se_from_ci(r3["HR_lo95"], r3["HR_hi95"]), nt.se_from_ci(r8["HR_lo95"], r8["HR_hi95"])]
            p = nt.pool_fixed_effect(b, s)
            cross.append({"outcome": outcome, "comparison": lab,
                          "NHANES_III_HR": r3["HR"], "NHANES_III_lo95": r3["HR_lo95"], "NHANES_III_hi95": r3["HR_hi95"],
                          "NHANES_0508_HR": r8["HR"], "NHANES_0508_lo95": r8["HR_lo95"], "NHANES_0508_hi95": r8["HR_hi95"],
                          "pooled_HR": np.exp(p["estimate"]), "pooled_lo95": np.exp(p["lo95"]), "pooled_hi95": np.exp(p["hi95"]),
                          "pooled_p": p["z_p"], "between_cohort_Q_p": p["Q_p"], "I2": p["I2"]})
        for (a, b), lab in zip([(0, 5), (5, 10), (10, 15)], ["0-5 y", "5-10 y", "10-15 y"]):
            r3 = tv3[(tv3["analysis"] == f"window {a}-{b} y") & (tv3["outcome"] == outcome)]
            if not r3.empty:
                r3 = r3.iloc[0]
                win_rows.append({"cohort": "NHANES III", "outcome": outcome, "window": lab, "HR": r3["HR"],
                                 "HR_lo95": r3["HR_lo95"], "HR_hi95": r3["HR_hi95"], "events": r3["events"]})
    for outcome in ["death_all", "death_cvd"]:
        for (a, b), lab in zip(WINDOWS, ["0-5 y", "5-10 y", "10-15 y"]):
            r8 = allres[(allres["analysis"] == f"window {a}-{b if b < 40 else 'end'} y, M2") & (allres["outcome"] == outcome)]
            if not r8.empty:
                r8 = r8.iloc[0]
                win_rows.append({"cohort": "NHANES 2005-2008", "outcome": outcome, "window": lab, "HR": r8["HR"],
                                 "HR_lo95": r8["HR_lo95"], "HR_hi95": r8["HR_hi95"], "events": r8["events"]})
    cross = pd.DataFrame(cross); windows_both = pd.DataFrame(win_rows)
    cross.to_csv(OUT_DIR / "cross_cohort_pooled.csv", index=False)
    windows_both.to_csv(OUT_DIR / "cross_cohort_windows.csv", index=False)
    display(cross.round(3)); display(windows_both.round(3))
else:
    windows_both = pd.DataFrame()
    print(f"{tv_file} not found - run the NHANES III notebook in this folder first to enable the comparison")

,outcome,comparison,NHANES_III_HR,NHANES_III_lo95,NHANES_III_hi95,NHANES_0508_HR,NHANES_0508_lo95,NHANES_0508_hi95,pooled_HR,pooled_lo95,pooled_hi95,pooled_p,between_cohort_Q_p,I2
0,death_all,primary definitions,1.315,1.098,1.576,1.315,1.072,1.613,1.315,1.149,1.506,0.000,0.997,0.0
1,death_all,levels >= 14 in both cohorts,1.364,1.129,1.649,1.306,1.026,1.661,1.342,1.156,1.557,0.000,0.778,0.0
2,death_cvd,primary definitions,1.455,1.068,1.981,1.376,1.013,1.867,1.414,1.138,1.757,0.002,0.800,0.0
3,death_cvd,levels >= 14 in both cohorts,1.556,1.082,2.238,1.552,1.128,2.135,1.554,1.223,1.975,0.000,0.993,0.0


,cohort,outcome,window,HR,HR_lo95,HR_hi95,events
0,NHANES III,death_all,0-5 y,1.612,1.239,2.098,722.0
1,NHANES III,death_all,5-10 y,1.092,0.835,1.430,924.0
2,NHANES III,death_all,10-15 y,1.094,0.804,1.489,938.0
3,NHANES III,death_cvd,0-5 y,1.492,1.059,2.103,329.0
4,NHANES III,death_cvd,5-10 y,1.439,0.930,2.225,370.0
5,NHANES III,death_cvd,10-15 y,0.868,0.495,1.524,379.0
6,NHANES 2005-2008,death_all,0-5 y,1.405,1.084,1.822,401.0
7,NHANES 2005-2008,death_all,5-10 y,1.254,0.956,1.645,564.0
8,NHANES 2005-2008,death_all,10-15 y,1.291,0.889,1.875,359.0
9,NHANES 2005-2008,death_cvd,0-5 y,1.155,0.707,1.886,138.0


## 9. Figures (submission style)

In [15]:
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator
if not windows_both.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.2), sharey=True)
    styles = {"NHANES III": dict(marker="o", color="#000000", mfc="#000000"),
              "NHANES 2005-2008": dict(marker="s", color="#777777", mfc="white")}
    order = ["0-5 y", "5-10 y", "10-15 y"]
    for ax, outcome, ylab in [(axes[0], "death_all", "Hazard ratio, all-cause death"),
                              (axes[1], "death_cvd", "Hazard ratio, CVD death")]:
        for k, (coh, st) in enumerate(styles.items()):
            sub = windows_both[(windows_both["cohort"] == coh) & (windows_both["outcome"] == outcome)].set_index("window").reindex(order)
            xs = np.arange(len(order)) + (k - 0.5) * 0.28
            ok = sub["HR"].notna().values
            ax.errorbar(xs[ok], sub["HR"][ok], yerr=[sub["HR"][ok] - sub["HR_lo95"][ok], sub["HR_hi95"][ok] - sub["HR"][ok]],
                        fmt=st["marker"], color=st["color"], mfc=st["mfc"], ms=7, capsize=3, lw=1.4, label=coh)
            for x_, (_, r) in zip(xs[ok], sub[ok].iterrows()):
                ax.annotate(f"{r['HR']:.2f}", (x_, r["HR"]), textcoords="offset points", xytext=(7, 0), va="center", fontsize=10)
        ax.axhline(1, color="#555555", lw=1, ls="--", label="HR = 1", zorder=0)
        ax.set_yscale("log")
        ticks = [0.125, 0.25, 0.5, 1, 2, 4, 8, 16]
        ax.yaxis.set_major_locator(FixedLocator(ticks)); ax.yaxis.set_minor_locator(NullLocator())
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
        ax.set_xticks(range(len(order))); ax.set_xticklabels(["0\u20135", "5\u201310", "10\u201315"])
        ax.set_xlabel("Years since examination"); ax.set_ylabel(ylab)
        lo = np.nanmin(windows_both["HR_lo95"]); hi = np.nanmax(windows_both["HR_hi95"])
        ax.set_ylim(max(0.1, lo / 1.15), min(20, hi * 1.15))
    axes[1].yaxis.set_tick_params(labelleft=True)
    h, l = axes[0].get_legend_handles_labels()
    order_l = [l.index(k) for k in ("NHANES III", "NHANES 2005-2008", "HR = 1") if k in l]
    fig.legend([h[i] for i in order_l], [l[i].replace("2005-2008", "2005\u20132008") for i in order_l],
               loc="lower center", ncol=3, frameon=False, fontsize=11)
    fig.tight_layout(rect=(0, 0.08, 1, 0.95), w_pad=3)
    nt.align_panel_letters(fig, [(axes[0], "A"), (axes[1], "B")])
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"fig_windows_two_cohorts.{ext}", dpi=300)
    plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
(axA, axB), (axC, axD) = axes
x = curves["years"].values
for ax, k0, k1, ylab in [(axA, "all_F0", "all_F1", "Cumulative mortality, %"),
                         (axB, "cause_CIF0", "cause_CIF1", "Cumulative CVD mortality, %")]:
    ax.step(x, curves[k0], where="post", color="#999999", lw=2.2, label="No retinopathy")
    ax.step(x, curves[k1], where="post", color="#000000", lw=2.2, label="Retinopathy")
    ax.set_xlim(0, CURVE_MAX); ax.set_ylim(bottom=0); ax.set_xticks(range(0, CURVE_MAX + 1, 2))
    ax.set_xlabel("Years since examination"); ax.set_ylabel(ylab)
for ax, k in [(axC, "all_RD"), (axD, "cause_RD")]:
    ax.fill_between(x, curves[f"{k}_lo95"], curves[f"{k}_hi95"], step="post", color="#DDDDDD", lw=0, label="95% CI")
    ax.step(x, curves[k], where="post", color="#000000", lw=2.2, label="Retinopathy minus no retinopathy")
    ax.axhline(0, color="#555555", lw=1, ls="--", label="No difference")
    for h in HORIZONS:
        v = jk.loc[f"{'all' if k == 'all_RD' else 'cause'}_RD_{h:g}", "estimate"]
        ax.plot([h], [v], "o", color="#000000", ms=6, clip_on=False, zorder=5)
        at_edge = h >= CURVE_MAX
        ax.annotate(f"{v:.1f}", (h, v), textcoords="offset points", xytext=(-4 if at_edge else 0, -11),
                    ha="right" if at_edge else "center", va="top", fontsize=11,
                    bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.9), zorder=6)
    ax.set_xlim(0, CURVE_MAX); ax.set_xticks(range(0, CURVE_MAX + 1, 2))
    ax.set_xlabel("Years since examination"); ax.set_ylabel("Difference, percentage points")
h1, l1 = axA.get_legend_handles_labels(); h2, l2 = axC.get_legend_handles_labels()
fig.legend(h1 + h2, l1 + l2, loc="lower center", ncol=3, frameon=False, fontsize=11)
fig.tight_layout(rect=(0, 0.06, 1, 0.97), h_pad=2.5, w_pad=2.0)
nt.align_panel_letters(fig, [(axA, "A"), (axB, "B"), (axC, "C"), (axD, "D")])
for ext in ("png", "pdf"):
    fig.savefig(FIG_DIR / f"fig_standardized_risk_0508.{ext}", dpi=300)
plt.close(fig)
print("figures saved to", FIG_DIR)

figures saved to c:\claude\nhanes3_eye_heart\output_0508\figures


## 10. Run log

In [16]:
RUNLOG.update({"windows": [list(w) for w in WINDOWS], "first_decade": list(FIRST_DECADE), "horizons": HORIZONS,
               "taus": TAUS, "c_horizon": C_HORIZON, "creatinine_calibration": {k: v for k, v in CREATININE_CAL.items()},
               "variance": "Cox: design-based Taylor linearization (strata + PSU); absolute measures: delete-one-PSU jackknife",
               "fit_errors": len(errors)})
with open(OUT_DIR / "run_log.json", "w") as fh:
    json.dump(RUNLOG, fh, indent=2, default=str)
print(json.dumps(RUNLOG, indent=2, default=str))

{
  "notebook": "nhanes0508_replication.ipynb",
  "osf_registration": "https://osf.io/j78ym",
  "test_mode": false,
  "notebook_code_sha256": "03febf8068a36bf7abd9f97e97d9ce94e99e8c69204c3c3e7e4b4e46b0674d83",
  "tools_sha256": "9fcdbfbd2c75436216dc355927a1e12c964df24daa141193b273f621c489e9a3",
  "python": "3.12.10",
  "pandas": "2.3.3",
  "numpy": "2.4.4",
  "lifelines": "0.30.3",
  "n_cohort": 5703,
  "deaths_all": 1449,
  "deaths_cvd": 440,
  "median_followup_reverse_km_years": 12.75,
  "benchmark_dr_in_diabetes_pct": 28.9,
  "covariates_model2": [
    "age",
    "female",
    "race_eth_2",
    "race_eth_3",
    "race_eth_4",
    "edu_hs",
    "edu_gt_hs",
    "bmi",
    "sbp",
    "bp_meds",
    "diabetes",
    "hba1c",
    "tc",
    "hdl",
    "egfr",
    "prior_mi",
    "smoke_former",
    "smoke_current"
  ],
  "verdicts": {
    "H1 (primary): first-decade HR > 1, all-cause": "supported: HR 1.31 (1.07-1.61)",
    "H1b: first-decade HR > 1, CVD": "supported: HR 1.38 (1.01-1.87)",